# Balanced teacher distillation — native-v3 → native-v4

Warm-start your completed **native-v3 rank-16 LoRA and pointer head**. V4 saves to its own output and recovery directory. Its first run starts a fresh optimizer and schedule; rerunning interrupted v4 resumes only v4's optimizer. No general-decision stages are repeated.

The [v3 audit](https://github.com/yxc20089/QPlusLearning/blob/main/labs/lab-01-kev-pacman/evaluation/v3-analysis-2026-10-07/README.md) found fewer immediate avoidable deaths, but zero clears and worse food efficiency. This experiment changes data selection: native evidence for immediate evasion, anticipatory escape, actual power-expiry hazards, retreat followed by food, productive junctions and sparse cleanup. Exact roles and overlaps are printed from the final manifest. A nearby ghost or low timer alone does not prove a bad alternative. Most labels come from independently verified winning teacher trajectories; archived v3 mistakes are identified only where their checkpoint, trace and state are bound. There are no live learner probes in this notebook.

**The balanced dataset is complete, validated and backed up to Drive on 8 October 2026.** This notebook imports that frozen export, checks its exact manifest fingerprint, validates the native evidence on CPU, then trains v4. Generation and teacher relabeling are already finished. Rank, observation/action protocol, CE objective and total update budget remain fixed. The main lab still has one assessed CP1.

| Primary targeted behavior | Train | Development |
| --- | ---: | ---: |
| Immediate evasion | 640 | 96 |
| Anticipatory escape | 64 | 16 |
| Actual native power-expiry evasion | 16 | 4 |
| Retreat followed by food | 512 | 48 |
| Productive routing | 1,408 | 128 |
| Sparse-pellet cleanup | 888 | 78 |
| Residual broad exposure | 132 | 14 |
| **Targeted roots** | **3,660** | **384** |

Training also includes 1,220 informative nearby decisions and 912 canonical Pac-Man replay records: **5,792 task requests**. Kev adds 304 generic replay records for **6,096 requests / 762 optimizer updates**. Development contains 384 roots plus 128 informative decisions: **512 requests**. Primary roles count each root once; overlap tags remain separate. All 11 qualified, recorded v3 mistakes are included. The 16/4 expiry allocation has limited coverage.

The full maze and four ghost states are present in every exported request. The CPU token audit found a maximum of 2,407 state tokens, below the 4,096-token limit. Training must still report its own truncation count. Data qualification does not establish learned-player improvement.

**Run in order:** setup and optimized preparation, Drive storage, verify the completed v3 parent, import the frozen data, CPU validation, then **Train v4 from the completed v3 weights**. The first v4 run initializes a fresh optimizer from v3's learned weights. Interrupted runs automatically resume their own latest v4 recovery. Comparison and interactive play follow completed training. Leave the required optimized backend enabled; the extra two-record training preflight remains optional.

For an already completed v4 checkpoint, run setup, storage, parent and dataset checks, then the training cell verifies its lineage and skips training. Continue to comparison or interactive play.

## Setup and Drive storage

Use the same RTX PRO 6000 Blackwell allocation and pinned optimized environment. Drive is enabled by default. Storage restores available v3/v4 checkpoints and their backed-up inputs. The import cell additionally looks in the completed local-generation handoff folder. Training uses the GPU; dataset import and provenance validation use the CPU. Source generation scripts remain in the repository history.

Checkpoint backups include the exact balanced training/development files, manifest and separate proof bundle. Wait for backup acknowledgments before replacing a runtime.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib, json, os, sys
TRAINING_PROFILE = 'memory_safe'  # Later stages; Stage 1 explicitly selects 4 x 2 below
LAB_DIR = Path.cwd() / 'pacman-kev-lab'
LAB_DIR.mkdir(exist_ok=True)
COURSE_REVISION = '27bfeb72cbc7c9e677a3af6be8477ec9f874e365'
FILES = {'api_client.py': 'd24f52b4bfb0c85fa1f4e67ab295eaa08e0f5654e71e6f0e0b030d7fb5347de6', 'pacman_lab.py': 'ebfa966b1a8f4bbaf90c98eacb11e77f2c18ec04158bee6744d1aee960307bc8', 'cloud_runtime.py': '9ad546991e4542d5d7d540b67d49218b0d3f6a656c209385a5111c0ff8deda52', 'training_monitor.py': '816f46da63ab7e936d55e85425ab9213145f2276fe6f2a260248307b45fbe681', 'lora_recovery.py': 'd01bc4f336b17948dc89fa8e931761c766fed979349a310d19f23e5ce0d6e274', 'checkpoint_backup.py': '1014a5616b5d17a58d89a0020e464ff393edc4bf9d4639aa77f48ef88b3d4937', 'optimized_training.py': '5c33fac17ec6ad6439193737aea650f85cdfb4445a0d346bb5bed10b858fcdfa', 'training-kernels.json': '24dff024433f61c4a571f16326f4f16c5666297e6514089e66e721b39765e56c', 'training_stages.py': '96fdd59285b8872b946523a88d95393decb73536322ca35e8f67f79cb2a7ef5c', 'training-stages.json': '9be232b730a9550a01011091b1abf3d79eaba3de8db5bdd455783fe5ae0e8ad0', 'games/arcade-engine.js': 'df5cf83c69810c641b65ccb533dd5da5b0ca6aca619db32090ad932fd8cd7e16', 'games/arcade-worker.js': 'acd16571bd330e2dc8033ab7c59977f192ae634d1eec4291db7e16a09b730daa', 'games/arcade-browser.js': 'e6f7fcba0e8d5c79573fe461e89efb428584acea594e9ec1e351e5d2964a0395', 'games/arcade-shell.html': 'b2606cdd679242c434570fd3850618041f79a8590f3b813106536c4d78341a0c', 'vendor/arcade-pacman/source.json': 'a0f21cf6009b7f7ca195fb8ce5f0730609767e7ced1c9b8cf29152f5bbe52545', 'vendor/arcade-pacman/source.zip': 'fd9ae0cbeebdfd798c609464e884f6e9c9b59673bd50083fbd0e32ea6c26a6e8', 'planner_data.py': '9b325f71bdc08419014b6e69b1541868d483c20efaaf79964892ac3887e65ca4', 'games/arcade-planner.js': 'f505a2b131465f0f170e16731863efe870e8549b50239f41d5a230dd57b7ff57', 'data/pacman-planner-v1.zip': 'a6f7e5282e888ede6a24f28cc7dffdb7d3cf51776ec5c9d6adefcf6e15e9e6a1', 'data/pacman-planner-v1-manifest.json': '3c22ede98985264b46410dc14026eaa22301b79a3f9061cf5063c739fc39b6e2', 'data/pacman-planner-v1-quality.json': 'db6ae0d8b730c7582f3f76e0c0177061d5449cfd55bc87ae3dd4c66277d54fc0', 'gameplay_benchmark.py': '62f205f9db11c7e067dbd07f609d0835fe92e1d2de3627aee2f7b641d2ed2008', 'benchmark-spec.json': '6be3f95e110422defe40c5c9007617f9e09f8c0e5c86a16b9107a73e52512cdc', 'games/benchmark-hooks.js': '9087a8ee70d8bb25c71a84abe8f614a4ea99f897ba345e36f4a70f520dca4069', 'games/benchmark-worker.cjs': '1d30685f16f5c55041fc5c50156da1119bd96c2ef382e09d4c7048939f7b1df7', 'games/arcade-teacher.js': '66d98198b77ad5364688006ed132804fae1ccc395fb4a4536980abbdafbe2f3b', 'teacher_validation.py': '0dfd7e7965a4837a7a989831802253d7f57a830283b241bf659ade7c876e8084', 'teacher-validation-spec.json': 'f5a57378e7020e543b14d4ab8ebaa0628097393307c2c493b96c0eb26d736780', 'teacher_data.py': '095a75cd23d2386987ce40d250b0fbd1979a01a97265ebc3a397aec749a847f0', 'evaluation/teacher-qualification.json': 'bd848daed094f60e3c84dc267f2569f644bf8ce04c324d79942bf0dd821dbce5', 'evaluation/teacher-qualification-replays.zip': 'bb21bcbf33ab5332c35a4f6022f8e4a1a4f70ef1546c0e5c3218142fc8d520c2', 'data/pacman-native-v2.zip': '8327bd809c86172283366cf018321e59ba91ceafc2d798555b28cb0aab6e7651', 'data/pacman-native-v2-manifest.json': '8cc0a7b26b010ff4bb1866e37f5bb3f5841c3c4a6f81d932d9ae52ba4971ae0f', 'data/pacman-native-v2-replays.zip': '6cfb6378951af88039f9503c13964a47cf83f7adaeb3d32245e69342b9652545', 'data/pacman-native-v2-generator.py': '31b7d5dcd4f750db0698f22535f13667f79e77e97cee525f373b8f99a9326be9', 'v3_data.py': '5220adf63df9326ac05c0581e7bffbb682ca1b4df4709152dd2a48e276c8d2d1', 'v4_data.py': '8a35c4f907d1391471b23ce11f6006141f931e6b00a00b408046e2a2ed0d6063', 'teacher_offline_data.py': 'e35f4ba38b45807637c8a4502326a31656c89071ee74fb2bf0a2d2f8e936ef48', 'scenario_catalog.py': '1962c676744570214a6d369ecd8f10a9343db16e039a85e2f73468c04f3570b4', 'collection_backup.py': '1f5b7b8f8412786ba504ed640720c931a44381e61b9557b457423d4bb125c625', 'hard_case_evaluation.py': 'cb2d68c5f1d3e34bcfeb18732858c5ceab23561c8f1eed62dfd1a5f360bb98e5', 'behavior_data.py': '83c2df6bf9f3f19ef3e39440fc04d3ce7b806e11e61e0a2d172f61e20950837f', 'behavior_scenarios.py': '435bced087eff5d32b011ce7f9371d8ba70ff30491caf6f8c9b3274c01350f52', 'recorded_behavior_scenarios.py': 'b5000d0af4f155be49d9c45d0898030df364ca5180256d86df1b07f462a7efa5', 'evaluation/v3-analysis-2026-10-07/teacher-reference.json': 'c3c06bb5a5de1cca6559da62206a1314aefa17c544c97bd7171b22903014c888'}
for name, expected_sha256 in FILES.items():
    target = LAB_DIR / name
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == expected_sha256:
        continue
    url = f'https://raw.githubusercontent.com/yxc20089/QPlusLearning/{COURSE_REVISION}/labs/lab-01-kev-pacman/{name}'
    with urlopen(url, timeout=60) as response:
        content = response.read()
    assert hashlib.sha256(content).hexdigest() == expected_sha256, f'Unexpected source: {name}'
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)
    print('Downloaded verified helper:', name, flush=True)
sys.path.insert(0, str(LAB_DIR))
# Refresh helpers when upgrading an existing notebook; keep checkpoints/logs.
if 'runtime' in globals():
    runtime.stop()
import importlib
importlib.invalidate_caches()
for name in ['cloud_runtime', 'training_monitor', 'lora_recovery', 'checkpoint_backup', 'optimized_training', 'training_stages', 'gameplay_benchmark', 'teacher_validation', 'teacher_data', 'planner_data', 'pacman_lab', 'api_client', 'v3_data', 'v4_data', 'collection_backup', 'hard_case_evaluation', 'behavior_data', 'behavior_scenarios', 'recorded_behavior_scenarios']:
    sys.modules.pop(name, None)
from cloud_runtime import CloudRuntime
from lora_recovery import latest_snapshot
from checkpoint_backup import new_run_directory
from training_stages import specifications, inspect_checkpoint, restore_checkpoint, backup_checkpoint
from pacman_lab import *
from teacher_data import prepare_dataset, PREFIX, RECIPE
from gameplay_benchmark import benchmark_gameplay, paired_gameplay
from api_client import call, distribution
ensure_node(LAB_DIR)
runtime = CloudRuntime(LAB_DIR, training_profile=TRAINING_PROFILE)
CHECKPOINT_ROOT = LAB_DIR / 'checkpoints'
INITIAL = CHECKPOINT_ROOT / 'kev-4b-initial'
DATES = CHECKPOINT_ROOT / 'kev-4b-dates'
DOCUMENTS = CHECKPOINT_ROOT / 'kev-4b-documents'
SKILLS = CHECKPOINT_ROOT / 'kev-4b-skills'
STAGES = specifications()
manifest = None  # Loaded with its teacher qualification receipt in CP1
GAME = notebook_game(action_version=2)
STAGE_OWNERS = {}
print('Prepared the native arcade engine, four ghosts and full-game benchmark. CP1 verifies qualified demonstrations.')

In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'tensorboard==2.20.0'])
LOG_DIR = LAB_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)
SHOW_TENSORBOARD = True
if SHOW_TENSORBOARD:
    from IPython import get_ipython
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic('load_ext', 'tensorboard')
        ip.run_line_magic('tensorboard', f'--logdir "{LOG_DIR}" --reload_interval 5')
    else:
        print('Outside a notebook, run: tensorboard --logdir', LOG_DIR)

In [ ]:
runtime.setup()

In [ ]:
RUN_TRAINING_PREFLIGHT = False  # Optional extra model checks; optimized bindings remain required
audit = runtime.prepare_training(run_training_preflight=RUN_TRAINING_PREFLIGHT)
print('Training suite records:', audit['suite_records'])
print('Trainable parameters:', sum(audit['trainable_parameters'].values()))
print('Optimized training verification:', json.dumps(runtime.training_preflight, indent=2))

In [ ]:
SAVE_TO_DRIVE = True  # True enables automatic backup and restore; no manual file copying
RESTORE_RECOVERY_ARCHIVE = None  # ZIP from the inspection/export cell; restore into an absent root
CHECKPOINT_ROOT = LAB_DIR / 'checkpoints'
DRIVE_BACKUP_ROOT = None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BACKUP_ROOT = Path('/content/drive/MyDrive/QPlusLearning/lab-01-kev-pacman/backups')
if RESTORE_RECOVERY_ARCHIVE is not None:
    restore_checkpoint(RESTORE_RECOVERY_ARCHIVE, CHECKPOINT_ROOT)
else:
    CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
print('Automatic Drive backup:', json.dumps(runtime.configure_backup(DRIVE_BACKUP_ROOT, CHECKPOINT_ROOT), indent=2))
print('Later-stage training profile:', runtime.training_profile, 'overrides:', runtime.selected_recipe({}))
print('Stage 1 selects batch 4 x accumulation 2 / row_budget 0 in its own training cell.')
print('Checkpoints and recovery:', CHECKPOINT_ROOT)

## Verify the completed native-v3 parent — CPU

Use `checkpoints/kev-4b-pacman-native-v3`, restored from its own Drive checkpoint backup. This experiment is bound to your completed v3 LoRA/head hash, not just a folder name. The cell checks the hash, rank, backbone and completed training metrics without loading the model onto the GPU. Skills, v2 or another v3 adapter cannot silently replace this parent.

In [ ]:
from behavior_data import validate_dataset, PREFIX as V4_PREFIX, RECIPE as V4_RECIPE
from hard_case_evaluation import evaluate_hard_cases, native_summary
from training_stages import checkpoint_fingerprint
from api_client import call_batch
V3 = CHECKPOINT_ROOT / 'kev-4b-pacman-native-v3'
V4 = CHECKPOINT_ROOT / 'kev-4b-pacman-native-v4'
V4_DATA = LAB_DIR / 'data'
V4_TRAINING = V4_DATA / f'{V4_PREFIX}-train.jsonl'
V4_DEVELOPMENT = V4_DATA / f'{V4_PREFIX}-development.jsonl'
V4_MANIFEST = V4_DATA / f'{V4_PREFIX}-manifest.json'
QUALIFICATION = LAB_DIR / 'evaluation/teacher-qualification.json'
EXPECTED_V3_SHA256 = 'c7c5c8326f74e86e161e05730db5004efdfd3b2351c6f2f9478581cfc13550b0'
EXPECTED_V4_MANIFEST_SHA256 = 'a1a151682f55e67de745165f8efdfc386e21a90e3eddfc0c002cbd9c40a38096'
V4_IMPORT_ROOT = Path('/content/drive/MyDrive/QPlusLearning/lab-01-kev-pacman/prework-backups/v4-balanced-generation-20261008')

def verify_v3_parent():
    required = ['training_config.json', 'training_metrics.json', 'run-evidence.json']
    if not all((V3 / name).is_file() for name in required):
        raise RuntimeError(f'Restore your completed native-v3 checkpoint to {V3}; run the storage cell first.')
    parent_sha = checkpoint_fingerprint(V3)
    if parent_sha != EXPECTED_V3_SHA256:
        raise ValueError('Native-v3 LoRA/head differs from the approved completed parent.')
    args = json.loads((V3 / 'training_config.json').read_text())['args']
    metrics = json.loads((V3 / 'training_metrics.json').read_text())
    evidence = json.loads((V3 / 'run-evidence.json').read_text())
    if (evidence.get('stage') != 'pacman' or args.get('lora') != 16
            or args.get('lora_targets') != 'all' or args.get('head_dim') != 256
            or args.get('base') != 'Qwen/Qwen3.5-4B-Base'
            or args.get('base_revision') != '1001bb4d826a52d1f399e183466143f4da7b741b'
            or metrics.get('optimizer_steps') != 762 or metrics.get('requested_records') != 6096
            or metrics.get('records_seen') != 6096 or metrics.get('truncated_records', 0)
            or metrics.get('rejected_records', 0)):
        raise ValueError('Native-v3 metadata does not describe the completed rank-16 parent.')
    return {'checkpoint': str(V3), 'checkpoint_sha256': parent_sha,
            'lora_rank': args['lora'], 'head_dim': args['head_dim'],
            'optimizer_steps': metrics['optimizer_steps'], 'base': args['base']}

V3_IDENTITY = verify_v3_parent()
print('Verified warm-start parent:', json.dumps(V3_IDENTITY, indent=2))
print('Separate v4 output:', V4)
print('Precomputed data handoff:', V4_IMPORT_ROOT)
print('Frozen dataset manifest SHA256:', EXPECTED_V4_MANIFEST_SHA256)
print('Selected training recipe:', V4_RECIPE)

## Import the completed balanced data and proof bundle

The completed Drive handoff contains train JSONL, development JSONL and the manifest. Its 133,513,786-byte evidence ZIP is stored as **two ordered parts** plus `pacman-native-v4-balanced-evidence.zip.parts.json`. The cell reconstructs the ZIP, checking every part and the final hash. It imports exactly four artifacts and never overwrites a conflicting local file. Matching restored files are reused.

The manifest must match the frozen 8 October export fingerprint shown above. You can change `V4_IMPORT_ROOT` to another mounted folder containing those same files. Missing or changed artifacts stop before training.

In [ ]:
# Import the four completed balanced artifacts from mounted Drive.
import tempfile
import shutil

def v4_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def v4_install_file(name, expected_sha256=None):
    destination = V4_DATA / name
    source = V4_IMPORT_ROOT / name
    if destination.is_symlink() or source.is_symlink():
        raise ValueError(f'Refusing symlink artifact: {name}')
    if destination.exists():
        digest = v4_file_sha256(destination)
        if expected_sha256 is not None and digest != expected_sha256:
            raise ValueError(f'Conflicting local artifact: {destination}; select a separate data directory.')
        if source.is_file() and v4_file_sha256(source) != digest:
            raise ValueError(f'Local and Drive artifacts differ: {name}; neither was overwritten.')
        return digest
    V4_DATA.mkdir(parents=True, exist_ok=True)
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(dir=V4_DATA, prefix=f'.{name}.', delete=False) as temporary:
            temporary_path = Path(temporary.name)
            if source.is_file():
                with source.open('rb') as handle:
                    shutil.copyfileobj(handle, temporary, length=8 * 1024 * 1024)
                source_sha = v4_file_sha256(source)
                expected_sha256 = expected_sha256 or source_sha
                if source_sha != expected_sha256:
                    raise ValueError(f'Drive artifact checksum differs: {name}')
            else:
                receipt_path = V4_IMPORT_ROOT / f'{name}.parts.json'
                if not name.endswith('-evidence.zip') or not receipt_path.is_file():
                    raise FileNotFoundError(f'Completed v4 handoff is missing {name} in {V4_IMPORT_ROOT}. No training started.')
                receipt = json.loads(receipt_path.read_text())
                parts = receipt.get('parts', [])
                if (receipt.get('schema') != 'ordered-file-parts-v1' or receipt.get('file') != name
                        or receipt.get('sha256') != expected_sha256 or not parts
                        or type(receipt.get('bytes')) is not int or receipt['bytes'] <= 0):
                    raise ValueError('Evidence parts receipt does not match the declared final artifact.')
                total = 0
                for index, part in enumerate(parts, 1):
                    if (part.get('file') != f'{name}.part-{index:03d}'
                            or type(part.get('bytes')) is not int or not 0 < part['bytes'] <= 64 * 1024 * 1024):
                        raise ValueError('Evidence parts are missing, reordered or outside the filename whitelist.')
                    part_path = V4_IMPORT_ROOT / part['file']
                    if part_path.is_symlink() or not part_path.is_file():
                        raise FileNotFoundError(f'Missing evidence part: {part_path}')
                    digest = hashlib.sha256(); size = 0
                    with part_path.open('rb') as handle:
                        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
                            digest.update(chunk); size += len(chunk); temporary.write(chunk)
                    if size != part['bytes'] or digest.hexdigest() != part.get('sha256'):
                        raise ValueError(f'Corrupt evidence part: {part_path.name}')
                    total += size
                    print('Verified evidence part:', part_path.name, flush=True)
                if total != receipt['bytes']:
                    raise ValueError('Reconstructed evidence size differs from its receipt.')
        digest = v4_file_sha256(temporary_path)
        if digest != expected_sha256:
            raise ValueError(f'Final artifact checksum differs: {name}')
        # Recheck before installing; preserve any file added during the copy.
        if destination.exists():
            if v4_file_sha256(destination) != digest:
                raise ValueError(f'Conflicting local artifact appeared during import: {name}')
        else:
            temporary_path.replace(destination)
        print('Verified balanced artifact:', name, flush=True)
        return digest
    finally:
        if temporary_path is not None and temporary_path.exists():
            temporary_path.unlink()

manifest_name = f'{V4_PREFIX}-manifest.json'
v4_install_file(manifest_name, EXPECTED_V4_MANIFEST_SHA256)
handoff_manifest = json.loads(V4_MANIFEST.read_text())
expected_names = {f'{V4_PREFIX}-train.jsonl', f'{V4_PREFIX}-development.jsonl'}
if (handoff_manifest.get('dataset') != V4_PREFIX or set(handoff_manifest.get('files', {})) != expected_names
        or handoff_manifest.get('evidence', {}).get('filename') != f'{V4_PREFIX}-evidence.zip'):
    raise ValueError('Unexpected balanced artifact names; no data files were imported.')
for name, digest in handoff_manifest['files'].items():
    v4_install_file(name, digest)
v4_install_file(handoff_manifest['evidence']['filename'], handoff_manifest['evidence']['sha256'])
print('Four balanced artifacts available:', V4_DATA)

## Strictly validate the selected data — CPU

Validation binds native source receipts, positive/negative branch evidence, teacher qualification, labels, legal options, deduplication, seed partitions, family caps, behavior quotas and the exact recipe. It also checks linked same-life windows: the escape follow-through floor requires food consumed **before** the window state and a grounded threat-escape parent. Ordinary later food does not count toward that floor. The first semantic validation can take several minutes on CPU. Reruns rehash all inputs and pinned source dependencies, reusing the successful check only while those bytes match. This receipt exists only in notebook memory; any changed input forces validation again.

The final task file must contain **3,660 targeted roots + 1,220 informative decisions + 912 unchanged expert replay = 5,792 Pac-Man requests**. Kev adds **304 generic replay requests**, making **6,096 requests / 762 updates**. Development has 384 targeted + 128 informative = 512 separate requests. Exact primary-role allocations and overlapping cohorts come from the validated manifest, not an assumed equal mixture.

In [ ]:
# Strict validation must pass before any v4 training or comparison.
from teacher_validation import source_hashes as v4_native_source_hashes
_V4_VALIDATION_RECEIPT = globals().get('_V4_VALIDATION_RECEIPT')

def v4_validation_binding():
    artifact_names = [f'{V4_PREFIX}-{suffix}' for suffix in
                      ['train.jsonl', 'development.jsonl', 'manifest.json', 'evidence.zip']]
    qualification = json.loads(QUALIFICATION.read_text())
    replay_name = qualification['replay_bundle']
    if not isinstance(replay_name, str) or Path(replay_name).name != replay_name:
        raise ValueError('Unexpected teacher qualification replay filename.')
    def optional_hash(path):
        path = Path(path)
        return v4_file_sha256(path) if path.is_file() else None
    return {'artifacts': {name: v4_file_sha256(V4_DATA / name) for name in artifact_names},
        'qualification': v4_file_sha256(QUALIFICATION),
        'qualification_replay': {replay_name: v4_file_sha256(QUALIFICATION.parent / replay_name)},
        'local_helpers': {name: v4_file_sha256(LAB_DIR / name) for name in FILES},
        'native_sources': v4_native_source_hashes(),
        'setup_pin': {'revision': COURSE_REVISION, 'files': dict(FILES)},
        'local_pin_files': {name: optional_hash(LAB_DIR / name) for name in ['notebook-source.json', 'source_lock.json']},
        'canonical_extracted': {name: optional_hash(LAB_DIR / 'data' / name) for name in
                                ['pacman-native-v2-train.jsonl', 'pacman-native-v2-development.jsonl']}}

def validated_v4_manifest():
    global _V4_VALIDATION_RECEIPT
    binding = v4_validation_binding()
    if binding['artifacts'][V4_MANIFEST.name] != EXPECTED_V4_MANIFEST_SHA256:
        _V4_VALIDATION_RECEIPT = None
        raise ValueError('Dataset manifest differs from the frozen 8 October balanced export.')
    receipt = json.loads(V4_MANIFEST.read_text())
    if (receipt['expected_training_requests'] != 6096 or receipt['expected_optimizer_steps'] != 762
            or receipt['recipe'] != V4_RECIPE
            or receipt['initialization'] != {'parent_checkpoint': 'kev-4b-pacman-native-v3', 'optimizer': 'fresh'}
            or receipt['counts']['train'] != {'targeted': 3660, 'informative': 1220, 'replay': 912}
            or receipt['counts']['development'] != {'targeted': 384, 'informative': 128}):
        raise ValueError('Balanced manifest has an unexpected parent, recipe or complete-data budget.')
    if _V4_VALIDATION_RECEIPT == binding:
        print("Rehashed data and sources match this runtime's successful semantic validation.", flush=True)
        return receipt
    # A failed new check must never replace a successful receipt.
    _V4_VALIDATION_RECEIPT = None
    print('Validating balanced labels, native proof bundle and quotas on CPU...', flush=True)
    if validate_dataset(V4_DATA, QUALIFICATION) is not True:
        raise ValueError('Balanced dataset validation did not pass.')
    after = v4_validation_binding()
    before_stable = {key: value for key, value in binding.items() if key != 'canonical_extracted'}
    after_stable = {key: value for key, value in after.items() if key != 'canonical_extracted'}
    if (before_stable != after_stable or any(value is not None and after['canonical_extracted'][name] != value
            for name, value in binding['canonical_extracted'].items())):
        raise ValueError('Validation inputs changed during the semantic check; rerun before training.')
    # Canonical train/development files may first appear when their verified ZIP is extracted.
    _V4_VALIDATION_RECEIPT = after
    return receipt

v4_manifest = validated_v4_manifest()
print(json.dumps({key: v4_manifest[key] for key in
    ['counts', 'role_plan', 'coverage', 'expected_training_requests', 'expected_optimizer_steps', 'initialization']}, indent=2))
print('Validated manifest SHA256:', v4_file_sha256(V4_MANIFEST))

## Train v4 from the completed v3 weights

Rank 16, all-module LoRA plus pointer head, CE, one epoch, lr 2e-5, batch 4 × accumulation 2, required optimized kernels, BF16 autocast and FP32 stored weights. The first run uses the v3 adapter/head with a fresh optimizer. A rerun resumes an interrupted **v4** recovery snapshot only; it does not reuse the v3 optimizer or redo generation. A completed v4 is reused only if its parent, exact training-data hash, recipe and full training metrics match.

Drive checkpoint backup saves the same input, development, manifest and proof bundle. Watch recovery and backup acknowledgments at step 1, every 100 steps or five minutes, and completion. The new data remain an experiment: no improvement is claimed until full-game evaluation.

In [ ]:
# Validate inputs and lineage, then train fresh v4 or resume only its recovery.
v4_manifest = validated_v4_manifest()
verify_v3_parent()
if not (V4 / 'run-evidence.json').is_file():
    RESUME_V4 = latest_snapshot(Path(str(V4) + '-recovery')) is not None
    print('Resume interrupted v4:', RESUME_V4)
    print('Selected v4 command:', runtime.finetuning_command(V4_TRAINING, V4, V3, recipe=V4_RECIPE))
    runtime.finetune(V4_TRAINING, V4, init_from=V3, steps=0, resume=RESUME_V4, recipe=V4_RECIPE)
else:
    print('Checking completed v4 checkpoint:', V4)
metrics = json.loads((V4 / 'training_metrics.json').read_text())
evidence = json.loads((V4 / 'run-evidence.json').read_text())
args = json.loads((V4 / 'training_config.json').read_text())['args']
if (metrics.get('optimizer_steps') != 762 or metrics.get('records_seen') != 6096
        or metrics.get('requested_records') != 6096 or metrics.get('truncated_records', 0)
        or metrics.get('rejected_records', 0) or evidence.get('stage') != 'pacman'
        or evidence.get('parent_checkpoint_sha256') != EXPECTED_V3_SHA256
        or evidence.get('training_data_sha256') != v4_manifest['files'][V4_TRAINING.name]
        or args.get('lora') != 16 or args.get('lora_targets') != 'all' or args.get('head_dim') != 256
        or any(args.get(key) != value for key, value in V4_RECIPE.items())):
    raise ValueError('V4 is incomplete or does not match the validated balanced experiment.')
print('Completed v4 LoRA/head SHA256:', checkpoint_fingerprint(V4))
print('V4 training metrics:', json.dumps(metrics, indent=2))

## Compare per-role development and full native games

Both models receive the same disjoint 512 development requests. Diagnostic agreement, CE and teacher-probability margin are reported by **primary behavior role** and overlapping cohort. These snapshot measures are secondary. The primary comparison runs the **unchanged 20 native starting games**, continuing through lives until clear, game-over or watchdog. No teacher or safety override changes the learner's choices. Save all trajectories and report clears/censoring, first-life food, pellet/native-frame efficiency, avoidable deaths with opportunity denominators, all-actions-fatal endpoints, dry intervals and sustained cycles.

These 20 starts are a **repeated reference benchmark used in earlier diagnosis, not a blind test**. They and their traces remain evaluation-only; they are never added to training. Use the separate development partition for further tuning. The archived, independently replay-verified frozen teacher reference on the same starts provides context, not a new teacher run. Each invocation saves a dated comparison directory. Pause interactive play first.

In [ ]:
# Per-role development is diagnostic; complete native games are primary.
v4_manifest = validated_v4_manifest()
verify_v3_parent()
comparison_run = new_run_directory(LAB_DIR / 'results', 'comparison-v3-v4',
    baseline_checkpoint=str(V3), fine_tuned_checkpoint=str(V4), action_version=2)
development = {}
gameplay = {}
for name, checkpoint in [('native-v3', V3), ('native-v4', V4)]:
    runtime.start(checkpoint)
    development[name] = evaluate_hard_cases(V4_DEVELOPMENT,
        predict_batch=lambda requests: [value for value, elapsed in call_batch('/v1/systemone', requests, concurrency=16)],
        model_info=runtime.active_model_info)
    (comparison_run / f'{name}-hard-development.json').write_text(json.dumps(development[name], indent=2))
    print(name, 'development by primary role:', json.dumps(development[name]['primary_roles'], indent=2))
    gameplay[name] = benchmark_gameplay(model_info=runtime.active_model_info, trace_dir=comparison_run / name)
    (comparison_run / f'{name}.json').write_text(json.dumps(gameplay[name], indent=2))
comparison = {'schema_version': 4, 'baseline_checkpoint': str(V3), 'fine_tuned_checkpoint': str(V4),
    'parent_checkpoint_sha256': EXPECTED_V3_SHA256,
    'dataset_manifest_sha256': v4_file_sha256(V4_MANIFEST),
    'reference_benchmark_interpretation': 'Repeated20 native starts used in prior diagnosis; evaluation-only, not a blind test.',
    'hard_development': development,
    'gameplay': {'general': gameplay['native-v3'], 'fine_tuned': gameplay['native-v4'],
                 'paired': paired_gameplay(gameplay['native-v3'], gameplay['native-v4'])},
    'behavior': {name: native_summary(report) for name, report in gameplay.items()},
    'recording_directory': str(comparison_run)}
from teacher_validation import source_hashes
teacher_reference = json.loads((LAB_DIR / 'evaluation/v3-analysis-2026-10-07/teacher-reference.json').read_text())
assert teacher_reference['simulation_protocol'] == gameplay['native-v3']['protocol']
assert teacher_reference['source_sha256'] == source_hashes()
assert teacher_reference['qualification_sha256'] == hashlib.sha256(QUALIFICATION.read_bytes()).hexdigest()
assert teacher_reference['options'] == json.loads(QUALIFICATION.read_text())['options']
assert teacher_reference['native_replays_verified']
comparison['teacher_reference'] = {'summary': teacher_reference['summary'],
    'source_sha256': teacher_reference['source_sha256'],
    'teacher_replay_bundle_sha256': teacher_reference['teacher_replay_bundle_sha256'],
    'interpretation': 'Archived frozen-teacher games on the same20 starts, independently replay-verified in the prior audit; not regenerated by this cell.'}
(comparison_run / 'comparison.json').write_text(json.dumps(comparison, indent=2))
(LAB_DIR / 'comparison-v3-v4.json').write_text(json.dumps(comparison, indent=2))
print('Native safety and food efficiency:', json.dumps(comparison['behavior'], indent=2))
print('All development probes and benchmark trajectories:', comparison_run)

## Interactive play with native-v4 — ungraded

Choose Kev and Start. The badge must identify `kev-4b-pacman-native-v4` and its LoRA/head hashes. The structured state, legal moves and native ghost/pellet mechanisms match evaluation. Each invocation saves a dated full trace. Pause before evaluation or backup.

In [ ]:
# Interactive play with the completed Pac-Man adapter
from IPython.display import display, HTML, JSON
PACMAN_PLAY_CHECKPOINT = CHECKPOINT_ROOT / 'kev-4b-pacman-native-v4'
# To inspect your existing v1 adapter, explicitly select CHECKPOINT_ROOT / 'kev-4b-pacman-planner-v1' instead.
if not (PACMAN_PLAY_CHECKPOINT / 'run-evidence.json').is_file():
    raise RuntimeError(f'Complete Stage 5 or restore its completed checkpoint to {PACMAN_PLAY_CHECKPOINT} before playing.')
if runtime.active_checkpoint is None or runtime.active_model_info()['checkpoint'] != str(PACMAN_PLAY_CHECKPOINT):
    runtime.start(PACMAN_PLAY_CHECKPOINT)
if not runtime.active_model_info()['pacman_fine_tuned']:
    raise RuntimeError('The active checkpoint is not recorded as Pac-Man fine-tuned. Load the completed Stage 5 checkpoint.')
fine_tuned_play_run = new_run_directory(LAB_DIR / 'results', 'interactive-fine-tuned',
                                       active_checkpoint=runtime.active_model_info(), seed=7, action_version=2)
fine_tuned_bridge = NotebookBridge(fine_tuned_play_run / 'fine-tuned-player-trace.jsonl', runtime.active_model_info)
print('Interactive state/action/probability trace:', fine_tuned_bridge.trace_path)
try:
    from google.colab import output
except ImportError:
    print('Kaggle/local: use the Python rollouts below; interactive callbacks require Colab.')
else:
    output.register_callback('pacman.decide', lambda state, selected_bridge=fine_tuned_bridge: JSON(selected_bridge.decide(state)))
    output.register_callback('pacman.model', lambda selected_bridge=fine_tuned_bridge: JSON(selected_bridge.model()))
    display(HTML(GAME))

## Final Drive backup: checkpoints, inputs, curves and full traces

Run before replacing the runtime. The session archive saves the imported manifest/proof bundle, training logs, per-role development predictions, complete native benchmark trajectories and interactive traces. Each checkpoint backup separately retains its exact training/development inputs and evidence. Wait for `Session backup complete` and an empty `checkpoint_errors` list.

In [ ]:
# Back up all available checkpoints, trajectories and traces to Google Drive
from pathlib import Path
import json
from google.colab import drive
from checkpoint_backup import backup_lab_to_drive
drive.mount('/content/drive')
runtime.stop()
DRIVE_LAB_ROOT = Path('/content/drive/MyDrive/QPlusLearning/lab-01-kev-pacman')
drive_receipt = backup_lab_to_drive(LAB_DIR, DRIVE_LAB_ROOT, checkpoint_root=CHECKPOINT_ROOT)
print(json.dumps(drive_receipt, indent=2))
if drive_receipt['checkpoint_errors']:
    raise RuntimeError('Traces were saved, but some checkpoint backups failed. See checkpoint_errors above and retry.')